# AB01 Jinwoo — knee exo (re-scaled ID)

One-off variant of `compare_processed_knee_exo_id.ipynb` for **AB01_Jinwoo** only.

- **Processed ID/IK/mocap**: `/media/metamobility3/Samsung_T52/Results/AB01_Jinwoo_knee/knee-exo/`
- **Telemetry**: `ab01_jinwoo_knee_*_exo_on.npz` (GPIO-synced **logged** model output vs GT)
- **RD GT**: ID time-shifted by encoder–IK xcorr lag (differs from GPIO-only alignment)
- **Vicon IK oracle**: offline TCN replay from processed IK (comparison only)
- **Encoder vs IK**: GPIO-synced angle agreement (§4)
- **Cache**: `analysis/cache/compare_processed_knee_exo_id.npz`


In [ ]:
import io
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.signal import butter, sosfilt, sosfiltfilt

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_SUBJECT_DIR = Path('/media/metamobility3/Samsung_T52/Results/AB01_Jinwoo_knee')
TELEMETRY_ROOT = PROJECT_ROOT

EXO_KIND = 'knee-exo'
JOINT = 'knee_angle_r'
JOINT_LABEL = 'Knee R'
MOMENT_COL = 'knee_angle_r_moment'
MOCAP_FS_HZ = 1000.0
LPF_CUTOFF_HZ, LPF_ORDER = 6.0, 4
GT_LPF_MODE = 'zero_phase'
MODEL_LPF_MODE = 'zero_phase'
TELEMETRY_PATTERN = 'ab01_jinwoo_knee_*_exo_on.npz'
PAPER_SUBDIR = 'knee_exo'

SUBJECT_TOKEN = 'ab01_jinwoo'
SUBJECT_NAME = 'AB01_Jinwoo'
SUBJECT_MASS_KG = 88.0

CACHE_DIR = PROJECT_ROOT / 'analysis' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / 'compare_processed_knee_exo_id.npz'
REPLAY_METRICS_CSV = CACHE_DIR / 'compare_processed_knee_exo_id_replay_metrics.csv'

LOAD_FROM_CACHE = False
SAVE_TO_CACHE = False  # same .npz path as multi-subject notebook — avoid overwriting unless intended

TRIM_START_SEC = 10.0
TRIM_END_SEC = 10.0

EXCLUDE_TRIALS = set()

GT_OFFSET_NMPKG_BY_STEM: Dict[str, float] = {}
GT_LAG_SAMPLES_BY_STEM: Dict[str, int] = {}

# RD: shift ID onto encoder clock using encoder–IK xcorr lag (post-GPIO).
ID_ALIGN_XCORR_STEMS = frozenset({'ab01_jinwoo_knee_0p8mps_rd_exo_on'})
ENCODER_IK_XCORR_MAX_LAG = 300

print(f'Exo: {EXO_KIND} | joint: {JOINT_LABEL}')
print(f'Processed subject dir: {PROCESSED_SUBJECT_DIR}')
print(f'Cache: {CACHE_PATH}')
print(f'LOAD_FROM_CACHE={LOAD_FROM_CACHE} | SAVE_TO_CACHE={SAVE_TO_CACHE}')


In [ ]:
def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * float(fs_hz)
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(arr) < 4:
        return arr
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, arr) if mode == 'zero_phase' else sosfilt(sos, arr)


def lpf_nan(x, fs_hz, cutoff_hz, order, mode):
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(3, order + 1):
        return arr
    if finite.all():
        return butter_lpf(arr, fs_hz, cutoff_hz, order, mode)
    idx = np.arange(arr.size, dtype=np.float64)
    filled = np.interp(idx, idx[finite], arr[finite])
    out = butter_lpf(filled, fs_hz, cutoff_hz, order, mode)
    out[~finite] = np.nan
    return out


def rmse_r2(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    return rmse, float(1.0 - ss_res / (ss_tot + 1e-12))


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def infer_fs_hz(time_s, default_fs=100.0):
    if time_s is None or len(time_s) < 3:
        return float(default_fs)
    dt = np.diff(np.asarray(time_s, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return float(1.0 / np.median(dt)) if dt.size else float(default_fs)


def _subject_token(stem: str) -> str:
    return '_'.join(stem.lower().split('_')[:2])


def subject_dir_from_stem(stem: str) -> Path:
    token = _subject_token(stem)
    if token != SUBJECT_TOKEN:
        raise FileNotFoundError(f'Not an AB01 trial: {stem}')
    if not PROCESSED_SUBJECT_DIR.is_dir():
        raise FileNotFoundError(PROCESSED_SUBJECT_DIR)
    return PROCESSED_SUBJECT_DIR


def trial_cond_speed(stem: str) -> Tuple[str, str]:
    parts = stem.lower().split('_')
    return parts[4].upper(), parts[3]


def trial_meta_from_stem(stem: str) -> Dict[str, object]:
    token = _subject_token(stem)
    subject = SUBJECT_NAME
    cond, speed = trial_cond_speed(stem)
    condition = f'{cond}_{speed}'
    speed_mps = float(speed.replace('mps', '').replace('p', '.'))
    return {
        'subject': subject,
        'task': cond,
        'speed': speed,
        'condition': condition,
        'speed_mps': speed_mps,
        'trial_key': f'{subject}::{condition}',
    }


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS_HZ):
    df = pd.read_csv(path, skiprows=[0, 1, 2, 4], header=0, low_memory=False, on_bad_lines='skip')
    df = df[pd.to_numeric(df['Frame'], errors='coerce').notna()].copy()
    df['jet'] = pd.to_numeric(df['jet'], errors='coerce')
    df = df.dropna(subset=['jet']).reset_index(drop=True)
    return np.arange(len(df)) / fs, df['jet'].to_numpy(dtype=float)


def normalize_gpio(gpio: np.ndarray) -> np.ndarray:
    arr = np.asarray(gpio, dtype=np.float64)
    g_range = arr.max() - arr.min()
    return arr if g_range <= 0 else (arr - arr.min()) / g_range


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> Optional[int]:
    above = np.asarray(signal, dtype=np.float64) > threshold
    for i in range(1, len(above)):
        if above[i - 1] and not above[i]:
            return i
    return None


def extract_gpio(npz) -> Tuple[np.ndarray, str]:
    for k in ('gpio_output', 'GPIO', 'gpio', 'trigger'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError('No GPIO key in npz')


def gpio_offset_s(t_exo, g_exo, t_mocap, g_mocap):
    idx_exo = first_falling_edge(g_exo)
    idx_mocap = first_falling_edge(normalize_gpio(g_mocap))
    if idx_exo is None or idx_mocap is None:
        return None, idx_exo, idx_mocap
    return float(t_mocap[idx_mocap] - t_exo[idx_exo]), idx_exo, idx_mocap


def resolve_trial_paths(trial_stem: str) -> Dict[str, Path]:
    cond, speed = trial_cond_speed(trial_stem)
    subj_dir = subject_dir_from_stem(trial_stem)
    return {
        'npz': TELEMETRY_ROOT / f'{trial_stem}.npz',
        'mocap': subj_dir / EXO_KIND / 'mocap' / f'{cond}_{speed}.csv',
        'id': subj_dir / EXO_KIND / 'id' / f'{cond}_{speed}_id.sto',
        'cond': cond,
        'speed': speed,
        'subject_dir': subj_dir,
    }


def load_gpio_sync_data(trial_stem: str) -> Dict:
    paths = resolve_trial_paths(trial_stem)
    for key in ('npz', 'mocap', 'id'):
        if not paths[key].exists():
            raise FileNotFoundError(f'Missing {key}: {paths[key]}')

    d = np.load(str(paths['npz']), allow_pickle=True)
    gpio, gpio_key = extract_gpio(d)
    t_raw = np.asarray(d['time'], dtype=np.float64) if 'time' in d.files else np.arange(len(gpio), dtype=np.float64)
    n = min(len(t_raw), len(gpio))
    t_raw, gpio = t_raw[:n], gpio[:n]

    t_mocap, gpio_mocap = parse_mocap_csv(paths['mocap'])
    offset_s, idx_exo, idx_mocap = gpio_offset_s(t_raw, gpio, t_mocap, gpio_mocap)

    return {
        'trial': trial_stem,
        'sync_method': 'gpio_falling_edge',
        'paths': paths,
        'npz': d,
        'gpio_key': gpio_key,
        'offset_s': offset_s,
        'idx_exo': idx_exo,
        'idx_mocap': idx_mocap,
        'fs_npz_hz': infer_fs_hz(t_raw),
        'fs_mocap_hz': infer_fs_hz(t_mocap, default_fs=MOCAP_FS_HZ),
        't_npz': t_raw,
        'gpio_npz': gpio,
        't_mocap': t_mocap,
        'gpio_mocap_raw': gpio_mocap,
        'gpio_mocap_norm': normalize_gpio(gpio_mocap),
        't_npz_aligned': t_raw + float(offset_s) if offset_s is not None else t_raw.copy(),
    }


def _exclude_set(values) -> set:
    if values is None:
        return set()
    return set(values)


def is_excluded_trial_key(trial_key: str) -> bool:
    return trial_key in _exclude_set(EXCLUDE_TRIALS)


def is_excluded_trial(stem: str) -> bool:
    if _subject_token(stem) != SUBJECT_TOKEN:
        return True
    return is_excluded_trial_key(trial_meta_from_stem(stem)['trial_key'])


def valid_telemetry_stem(stem: str) -> bool:
    return _subject_token(stem) == SUBJECT_TOKEN and not is_excluded_trial(stem)



def filter_excluded_trials(trial_data: Dict[str, Dict]) -> Dict[str, Dict]:
    kept = {stem: wave for stem, wave in trial_data.items() if not is_excluded_trial(stem)}
    dropped = len(trial_data) - len(kept)
    if dropped:
        print(
            f'Excluded {dropped} trial(s) | subjects={sorted(_exclude_set(EXCLUDE_SUBJECTS))} '
            f'| trials={sorted(_exclude_set(EXCLUDE_TRIALS))}'
        )
    return kept


def analysis_trim_mask(
    t: np.ndarray,
    trim_start_s: float = TRIM_START_SEC,
    trim_end_s: float = TRIM_END_SEC,
) -> np.ndarray:
    """Post-sync mask: drop first/last N seconds of each trial."""
    t_rel = np.asarray(t, dtype=np.float64) - np.nanmin(t)
    t_end = float(np.nanmax(t_rel))
    return (t_rel >= float(trim_start_s)) & (t_rel <= t_end - float(trim_end_s))


def model_out_nmpkg_lpf(raw: np.ndarray, fs_hz: float) -> np.ndarray:
    return lpf_nan(raw, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, MODEL_LPF_MODE)


def enrich_model_out_nmpkg(wave: Dict) -> Dict:
    if 'model_out_nmpkg' in wave:
        return wave
    fs_hz = float(wave.get('fs_hz', infer_fs_hz(wave['t'])))
    out = dict(wave)
    out['fs_hz'] = fs_hz
    out['model_out_nmpkg'] = model_out_nmpkg_lpf(out['model_out_nmpkg_raw'], fs_hz)
    return out


def save_processed_exo_cache(trial_data: Dict[str, Dict], path: Path = CACHE_PATH) -> None:
    if not trial_data:
        raise RuntimeError('TRIAL_DATA is empty')
    payload = {'trial_keys': np.array(sorted(trial_data.keys()), dtype=object)}
    for trial_key, d in trial_data.items():
        p = str(trial_key)
        payload[f'{p}__t'] = np.asarray(d['t'], dtype=np.float64)
        payload[f'{p}__gt_nmpkg'] = np.asarray(d['gt_nmpkg'], dtype=np.float64)
        payload[f'{p}__model_out_nmpkg_raw'] = np.asarray(d['model_out_nmpkg_raw'], dtype=np.float64)
        payload[f'{p}__model_out_nmpkg'] = np.asarray(d['model_out_nmpkg'], dtype=np.float64)
        payload[f'{p}__meta'] = np.array([
            d['joint'], d['moment_col'], d['mass_kg'], d['offset_s'],
            d['applied_key'], d['gpio_key'], d['model_out_key'], d['trial_key'],
        ], dtype=object)
    np.savez_compressed(str(path), **payload)
    print(f'Saved {len(trial_data)} trials → {path}')


def load_processed_exo_cache(path: Path = CACHE_PATH) -> Dict[str, Dict]:
    if not path.exists():
        raise FileNotFoundError(f'Cache not found: {path}')
    data = np.load(str(path), allow_pickle=True)
    trial_data: Dict[str, Dict] = {}
    for trial_key in data['trial_keys']:
        p = str(trial_key)
        meta = data[f'{p}__meta']
        wave = {
            'trial': p,
            'joint': str(meta[0]),
            'moment_col': str(meta[1]),
            'mass_kg': float(meta[2]),
            'offset_s': float(meta[3]),
            'applied_key': str(meta[4]),
            'gpio_key': str(meta[5]),
            'model_out_key': str(meta[6]),
            'trial_key': str(meta[7]),
            't': np.asarray(data[f'{p}__t'], dtype=np.float64),
            'gt_nmpkg': np.asarray(data[f'{p}__gt_nmpkg'], dtype=np.float64),
            'model_out_nmpkg_raw': np.asarray(data[f'{p}__model_out_nmpkg_raw'], dtype=np.float64),
        }
        if f'{p}__model_out_nmpkg' in data.files:
            wave['model_out_nmpkg'] = np.asarray(data[f'{p}__model_out_nmpkg'], dtype=np.float64)
        trial_data[p] = enrich_model_out_nmpkg(wave)
    print(f'Loaded {len(trial_data)} trials from {path}')
    return trial_data

def extract_model_out_nmpkg_raw(npz) -> Tuple[np.ndarray, str]:
    for k in ('model_out_nmpkg_raw', 'model_out_nmpkg'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError(f'No model output key; available: {sorted(npz.files)}')


def extract_applied_cmd_nm(npz) -> Tuple[np.ndarray, str]:
    for k in ('cmd_R', 'cmd_L'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError(f'No cmd_R/cmd_L; available: {sorted(npz.files)}')




def _shift_samples_1d(x: np.ndarray, lag_samples: int) -> np.ndarray:
    """Shift series in time by lag_samples (positive => signal appears earlier)."""
    arr = np.asarray(x, dtype=np.float64)
    out = np.full_like(arr, np.nan)
    lag = int(lag_samples)
    if lag > 0:
        if lag < len(arr):
            out[:-lag] = arr[lag:]
    elif lag < 0:
        lag = -lag
        if lag < len(arr):
            out[lag:] = arr[:-lag]
    else:
        out = arr.copy()
    return _fill_nan_1d(out)


def _load_vicon_knee_ik_deg(stem: str) -> Tuple[np.ndarray, np.ndarray]:
    paths = resolve_trial_paths(stem)
    ik_path = paths['subject_dir'] / EXO_KIND / 'ik' / f"{paths['cond']}_{paths['speed']}_ik.mot"
    if not ik_path.is_file():
        raise FileNotFoundError(ik_path)
    cols, data = read_sto(ik_path)
    t_mocap = data[:, cols.index('time')].astype(np.float64)
    knee_deg = data[:, cols.index('knee_angle_r')].astype(np.float64)
    return t_mocap, knee_deg


def _load_synced_encoder_angle_deg(trial_stem: str, sync: Dict, n: int) -> Tuple[np.ndarray, str]:
    d = sync['npz']
    t_npz = np.asarray(sync['t_npz'][:n], dtype=np.float64)
    wave_stub = {'t': sync['t_npz_aligned'][:n], 'offset_s': sync['offset_s']}
    for key in ('model_in_knee_angle_raw', 'knee_angle_r'):
        if key in d.files:
            enc_raw = np.asarray(d[key][:n], dtype=np.float64)
            enc_sync = sync_to_wave_t(t_npz, enc_raw, wave_stub, t_src_on_npz_clock=True)
            return np.rad2deg(enc_sync), key
    raise KeyError(f'No encoder angle in {trial_stem}')


def _load_synced_vicon_ik_angle_deg(trial_stem: str, sync: Dict, n: int) -> np.ndarray:
    t_mocap, knee_deg = _load_vicon_knee_ik_deg(trial_stem)
    wave_stub = {'t': sync['t_npz_aligned'][:n], 'offset_s': sync['offset_s']}
    return sync_to_wave_t(t_mocap, knee_deg, wave_stub, t_src_on_npz_clock=False)


def compute_encoder_ik_xcorr_lag_samples(
    trial_stem: str,
    sync: Dict,
    n: int,
    *,
    max_lag: int = ENCODER_IK_XCORR_MAX_LAG,
) -> Tuple[int, str]:
    """Best lag aligning encoder to Vicon IK on GPIO-synced timeline (positive => encoder lags IK)."""
    enc_deg, enc_key = _load_synced_encoder_angle_deg(trial_stem, sync, n)
    vicon_deg = _load_synced_vicon_ik_angle_deg(trial_stem, sync, n)
    trim_m = analysis_trim_mask(sync['t_npz_aligned'][:n])
    best_lag = 0
    best_score = -np.inf
    for lag in range(-int(max_lag), int(max_lag) + 1):
        shifted = _shift_samples_1d(enc_deg, lag)
        s = np.asarray(shifted, dtype=np.float64)
        v = np.asarray(vicon_deg, dtype=np.float64)
        m = trim_m
        s, v = s[m], v[m]
        mm = np.isfinite(s) & np.isfinite(v)
        if mm.sum() < 100:
            continue
        score = float(np.corrcoef(s[mm], v[mm])[0, 1])
        if score > best_score:
            best_score = score
            best_lag = int(lag)
    return best_lag, enc_key


def apply_gt_trial_corrections(stem: str, gt_nmpkg: np.ndarray) -> Tuple[np.ndarray, Dict[str, object]]:
    """Per-trial GT lag / offset after GPIO-synced ID − cmd construction."""
    gt = np.asarray(gt_nmpkg, dtype=np.float64).copy()
    meta: Dict[str, object] = {
        'gt_offset_nmpkg': 0.0,
        'gt_lag_samples': 0,
        'gt_correction_note': '',
    }
    notes: List[str] = []
    lag = int(GT_LAG_SAMPLES_BY_STEM.get(stem, 0))
    if lag != 0:
        gt = _shift_samples_1d(gt, lag)
        meta['gt_lag_samples'] = lag
        notes.append(f'lag {lag:+d} samples')
    offset = float(GT_OFFSET_NMPKG_BY_STEM.get(stem, 0.0))
    if offset != 0.0:
        gt = gt + offset
        meta['gt_offset_nmpkg'] = offset
        notes.append(f'offset {offset:+.2f} N·m/kg')
    meta['gt_correction_note'] = '; '.join(notes)
    return gt, meta


def load_moment_waveforms(trial_stem: str, sync: Dict) -> Dict:
    """GPIO-synced GT and telemetry waveforms for one trial."""
    paths = sync['paths']
    mass = SUBJECT_MASS_KG
    d = sync['npz']

    applied_nm, applied_key = extract_applied_cmd_nm(d)
    model_out_raw, model_out_key = extract_model_out_nmpkg_raw(d)

    n = min(len(sync['t_npz']), len(applied_nm), len(model_out_raw))
    t_aligned = sync['t_npz_aligned'][:n].astype(np.float64)
    applied_nm = applied_nm[:n]
    model_out_nmpkg_raw = np.asarray(model_out_raw[:n], dtype=np.float64)
    fs_hz = infer_fs_hz(sync['t_npz'][:n])

    gpio_offset_s = float(sync['offset_s'])
    gpio_offset_samples = gpio_offset_s * fs_hz
    encoder_ik_lag = 0
    encoder_key = ''
    id_time_shift_samples = 0
    id_time_shift_note = ''

    if trial_stem in ID_ALIGN_XCORR_STEMS:
        encoder_ik_lag, encoder_key = compute_encoder_ik_xcorr_lag_samples(trial_stem, sync, n)
        id_time_shift_samples = int(encoder_ik_lag)
        id_time_shift_note = (
            f'ID shifted +{encoder_ik_lag} samples ({encoder_ik_lag / fs_hz * 1000.0:+.0f} ms) '
            f'vs GPIO-only (GPIO={gpio_offset_s:+.3f}s, xcorr lag={encoder_ik_lag:+d})'
        )

    cols, id_data = read_sto(paths['id'])
    t_id = id_data[:, cols.index('time')]
    id_moment_nm = id_data[:, cols.index(MOMENT_COL)]

    t_id_query = t_aligned + id_time_shift_samples / fs_hz
    id_nm_raw = np.interp(t_id_query, t_id, id_moment_nm, left=np.nan, right=np.nan)
    id_nmpkg_raw = id_nm_raw / mass
    applied_nmpkg_raw = applied_nm / mass

    net_raw_nmpkg = id_nmpkg_raw - applied_nmpkg_raw
    gt_nmpkg = lpf_nan(net_raw_nmpkg, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, GT_LPF_MODE)
    gt_nmpkg, gt_meta = apply_gt_trial_corrections(trial_stem, gt_nmpkg)
    if id_time_shift_note:
        gt_meta['gt_correction_note'] = (
            f"{gt_meta['gt_correction_note']}; {id_time_shift_note}".strip('; ')
            if gt_meta['gt_correction_note'] else id_time_shift_note
        )
    model_out_nmpkg = model_out_nmpkg_lpf(model_out_nmpkg_raw, fs_hz)

    meta = trial_meta_from_stem(trial_stem)
    return {
        'trial': trial_stem,
        'trial_key': meta['trial_key'],
        'joint': JOINT,
        't': t_aligned,
        'gt_nmpkg': gt_nmpkg,
        'model_out_nmpkg_raw': model_out_nmpkg_raw,
        'model_out_nmpkg': model_out_nmpkg,
        'applied_key': applied_key,
        'model_out_key': model_out_key,
        'mass_kg': mass,
        'moment_col': MOMENT_COL,
        'fs_hz': fs_hz,
        'gt_offset_nmpkg': gt_meta['gt_offset_nmpkg'],
        'gt_lag_samples': gt_meta['gt_lag_samples'],
        'gt_correction_note': gt_meta['gt_correction_note'],
        'gpio_offset_s': gpio_offset_s,
        'gpio_offset_samples': gpio_offset_samples,
        'encoder_ik_xcorr_lag_samples': encoder_ik_lag,
        'encoder_angle_key': encoder_key,
        'id_time_shift_samples': id_time_shift_samples,
        'id_time_shift_note': id_time_shift_note,
    }




def _fill_nan_1d(x: np.ndarray) -> np.ndarray:
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.all() or not finite.any():
        return arr
    arr[~finite] = np.interp(np.flatnonzero(~finite), np.flatnonzero(finite), arr[finite])
    return arr


def sync_to_wave_t(
    t_src: np.ndarray,
    y_src: np.ndarray,
    wave: Dict,
    *,
    t_src_on_npz_clock: bool = False,
) -> np.ndarray:
    """Resample onto GPIO-aligned `wave['t']` (same clock as ID GT)."""
    t_aligned = np.asarray(wave['t'], dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)
    t_src = np.asarray(t_src, dtype=np.float64)
    if t_src_on_npz_clock:
        n = int(min(len(t_aligned), len(t_src), len(y_src)))
        t_aligned = t_aligned[:n]
        t_src = t_src[:n]
        y_src = y_src[:n]
        t_ref = t_src + float(wave['offset_s'])
    else:
        t_ref = t_src
    y_sync = np.interp(t_aligned, t_ref, y_src, left=np.nan, right=np.nan)
    return _fill_nan_1d(y_sync)


def process_trial_waveforms(trial_stem: str):
    try:
        sync = load_gpio_sync_data(trial_stem)
    except FileNotFoundError as exc:
        return None, str(exc)
    if sync['offset_s'] is None:
        return None, 'no GPIO falling edge'
    wave = load_moment_waveforms(trial_stem, sync)
    wave['offset_s'] = float(sync['offset_s'])
    wave['gpio_key'] = sync['gpio_key']
    return wave, None

print('Helpers ready.')


## 1. Batch processing (GPIO sync)


In [ ]:
CANDIDATES = sorted(
    p for p in TELEMETRY_ROOT.glob(TELEMETRY_PATTERN)
    if valid_telemetry_stem(p.stem)
)
print(f'Found {len(CANDIDATES)} AB01 telemetry files')

WARNINGS: List[str] = []
TRIAL_DATA: Dict[str, Dict] = {}

if LOAD_FROM_CACHE:
    TRIAL_DATA = load_processed_exo_cache(CACHE_PATH)
    TRIAL_DATA = {k: v for k, v in TRIAL_DATA.items() if _subject_token(k) == SUBJECT_TOKEN}
else:
    for npz_path in CANDIDATES:
        stem = npz_path.stem
        try:
            wave, err = process_trial_waveforms(stem)
            if wave is None:
                WARNINGS.append(f'[WARN] {stem}: {err}')
                print(f'SKIP {stem}: {err}')
                continue
            TRIAL_DATA[stem] = wave
            print(f"OK  {stem} | offset={wave['offset_s']:+.3f}s | n={len(wave['t'])}")
        except Exception as exc:
            WARNINGS.append(f'[WARN] {stem}: {exc}')
            print(f'FAIL {stem}: {exc}')

    if SAVE_TO_CACHE and TRIAL_DATA:
        save_processed_exo_cache(TRIAL_DATA)

TRIAL_DATA = filter_excluded_trials(TRIAL_DATA)
TRIAL_DATA = {stem: enrich_model_out_nmpkg(wave) for stem, wave in TRIAL_DATA.items()}

for w in WARNINGS:
    print(w)
print('\nTiming alignment (GPIO vs encoder–IK xcorr):')
for stem, wave in sorted(TRIAL_DATA.items()):
    gpio_s = float(wave.get('gpio_offset_s', wave.get('offset_s', np.nan)))
    gpio_n = float(wave.get('gpio_offset_samples', gpio_s * wave.get('fs_hz', np.nan)))
    xcorr_n = int(wave.get('encoder_ik_xcorr_lag_samples', 0))
    id_shift = int(wave.get('id_time_shift_samples', 0))
    fs = float(wave.get('fs_hz', np.nan))
    print(
        f"  {stem}: GPIO={gpio_s:+.3f}s ({gpio_n:+.1f} samples) | "
        f"encoder–IK xcorr={xcorr_n:+d} samples ({xcorr_n / fs * 1000:+.0f} ms) | "
        f"ID shift={id_shift:+d} samples"
    )
    if wave.get('id_time_shift_note'):
        print(f"    → {wave['id_time_shift_note']}")
print(f'\nLoaded {len(TRIAL_DATA)} trials')


In [18]:
## 2. GPIO sync QC (optional)

GPIO_PALETTE = {'mocap': '#90A4AE', 'telemetry': '#FF9800'}

def draw_gpio_sync(sync: Dict, window_s: float = 4.0) -> None:
    t_edge_mocap = float(sync['t_mocap'][sync['idx_mocap']])
    t_edge_npz = float(sync['t_npz'][sync['idx_exo']])
    t0, t1 = t_edge_mocap - 0.5, t_edge_mocap + window_s
    fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
    m_npz = (sync['t_npz'] >= t_edge_npz - 0.5) & (sync['t_npz'] <= t_edge_npz + window_s)
    m_mocap = (sync['t_mocap'] >= t0) & (sync['t_mocap'] <= t1)
    axs[0].plot(sync['t_npz'][m_npz], sync['gpio_npz'][m_npz], color=GPIO_PALETTE['telemetry'], lw=1.4, ls='--', label=f"Telemetry ({sync['gpio_key']})")
    axs[0].plot(sync['t_mocap'][m_mocap], sync['gpio_mocap_norm'][m_mocap], color=GPIO_PALETTE['mocap'], lw=1.2, label='Mocap jet (norm)')
    axs[0].axvline(t_edge_npz, color=GPIO_PALETTE['telemetry'], ls=':')
    axs[0].axvline(t_edge_mocap, color=GPIO_PALETTE['mocap'], ls=':')
    axs[0].set_ylabel('Amplitude (a.u.)'); axs[0].set_title('Before sync'); axs[0].legend(); axs[0].grid(alpha=0.25)
    m_a_npz = (sync['t_npz_aligned'] >= t0) & (sync['t_npz_aligned'] <= t1)
    axs[1].plot(sync['t_mocap'][m_mocap], sync['gpio_mocap_norm'][m_mocap], color=GPIO_PALETTE['mocap'], lw=1.2, label='Mocap jet (norm)')
    axs[1].plot(sync['t_npz_aligned'][m_a_npz], sync['gpio_npz'][m_a_npz], color=GPIO_PALETTE['telemetry'], lw=1.4, ls='--', label='Telemetry shifted')
    axs[1].axvline(t_edge_mocap, color='black', ls=':')
    axs[1].set_xlabel('Mocap time (s)'); axs[1].set_ylabel('Amplitude (a.u.)')
    axs[1].set_title(f'After sync | offset {sync["offset_s"]:+.4f} s')
    axs[1].legend(); axs[1].grid(alpha=0.25)
    fig.suptitle(f"{sync['trial']} | GPIO sync", y=1.01)
    fig.tight_layout(); plt.show()

gpio_trial_dd = widgets.Dropdown(options=sorted(TRIAL_DATA), description='Trial:')
gpio_window = widgets.FloatSlider(value=4.0, min=1.0, max=15.0, step=0.5, description='Window (s):')
gpio_out = widgets.Output()

def _draw_gpio(*_):
    with gpio_out:
        gpio_out.clear_output(wait=True)
        draw_gpio_sync(load_gpio_sync_data(gpio_trial_dd.value), gpio_window.value)

gpio_trial_dd.observe(_draw_gpio, names='value')
gpio_window.observe(_draw_gpio, names='value')
display(widgets.VBox([widgets.HBox([gpio_trial_dd, gpio_window]), gpio_out]))
_draw_gpio()


## 3. Logged output vs GT (+ Vicon IK oracle)

Uses on-device **logged** `model_out_nmpkg` from telemetry (batch §1), not offline encoder replay.


In [ ]:
import inspect
import sys

import torch
import yaml

if not TRIAL_DATA:
    raise RuntimeError('No trials loaded. Run batch processing first.')

CTRL_CFG_PATH = PROJECT_ROOT / 'knee-exo-ctrl' / 'cfg' / 'final.yaml'
VICON_ORACLE_CKPT = PROJECT_ROOT / 'runs' / '0512_ik_id_knee_causal_in_zero_out' / 'best_model.pt'

_ctrl_cfg = yaml.safe_load(CTRL_CFG_PATH.read_text()) if CTRL_CFG_PATH.is_file() else {}
VICON_ANGLE_LPF_HZ = float(_ctrl_cfg.get('angle_lpf_hz', 6.0))
VICON_ANGLE_LPF_ORDER = int(_ctrl_cfg.get('angle_lpf_order', 4))
TRAINING_VEL_LPF_HZ = 15.0
TRAINING_VEL_LPF_ORDER = 4

sys.path.insert(0, str(PROJECT_ROOT))
from model import TCN  # noqa: E402


class _CausalLowPass:
    def __init__(self, fs_hz: float, cutoff_hz: float, order: int = 4):
        self.order = max(1, int(order))
        if cutoff_hz <= 0.0:
            self.alpha = 1.0
        else:
            dt = 1.0 / float(fs_hz)
            tau = 1.0 / (2.0 * np.pi * float(cutoff_hz))
            self.alpha = dt / (tau + dt)
        self.state = [0.0] * self.order
        self.initialized = False

    def update(self, x: float) -> float:
        x = float(x)
        if not self.initialized:
            self.state = [x] * self.order
            self.initialized = True
            return x
        y = x
        for i in range(self.order):
            self.state[i] = self.state[i] + self.alpha * (y - self.state[i])
            y = self.state[i]
        return float(y)


def apply_causal_lpf_series(x: np.ndarray, fs_hz: float, cutoff_hz: float, order: int) -> np.ndarray:
    lpf = _CausalLowPass(fs_hz, cutoff_hz, order)
    return np.asarray([lpf.update(float(v)) for v in np.asarray(x, dtype=np.float64)], dtype=np.float32)


def _tcn_ctor_kwargs(cfg: dict) -> dict:
    allowed = {k for k in inspect.signature(TCN.__init__).parameters if k != 'self'}
    return {k: v for k, v in cfg.items() if k in allowed}


def _vicon_ik_path(stem: str) -> Path:
    paths = resolve_trial_paths(stem)
    ik_path = paths['subject_dir'] / EXO_KIND / 'ik' / f"{paths['cond']}_{paths['speed']}_ik.mot"
    if not ik_path.is_file():
        raise FileNotFoundError(ik_path)
    return ik_path


def _load_vicon_knee_ik(stem: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    cols, data = read_sto(_vicon_ik_path(stem))
    t_mocap = data[:, cols.index('time')].astype(np.float64)
    knee_rad = np.deg2rad(data[:, cols.index('knee_angle_r')].astype(np.float64))
    fs_mocap = infer_fs_hz(t_mocap, default_fs=100.0)
    return t_mocap, knee_rad, fs_mocap


@torch.no_grad()
def run_knee_tcn_inference(
    model: TCN,
    angle: np.ndarray,
    vel: np.ndarray,
    window_size: int,
    device: str,
) -> np.ndarray:
    angle = np.asarray(angle, dtype=np.float32)
    vel = np.asarray(vel, dtype=np.float32)
    n = int(min(len(angle), len(vel)))
    pred = np.zeros(n, dtype=np.float32)
    model.eval()
    for t in range(n):
        start = max(0, t - window_size + 1)
        valid = t - start + 1
        x = np.zeros((2, window_size), dtype=np.float32)
        x[0, -valid:] = angle[start : t + 1]
        x[1, -valid:] = vel[start : t + 1]
        xt = torch.from_numpy(x).unsqueeze(0).to(device=device, dtype=torch.float32)
        y = model(xt)
        pred[t] = float(y[0, 0, -1].item())
    return pred


def _vicon_ik_velocity_spline(t_mocap: np.ndarray, knee_rad: np.ndarray) -> np.ndarray:
    from scipy.interpolate import splrep, splev

    t_mocap = np.asarray(t_mocap, dtype=np.float64)
    knee_rad = np.asarray(knee_rad, dtype=np.float64)
    if len(t_mocap) < 5:
        dt = 1.0 / infer_fs_hz(t_mocap)
        vel = np.zeros_like(knee_rad)
        if len(knee_rad) > 1:
            vel[1:] = (knee_rad[1:] - knee_rad[:-1]) / dt
        return vel
    tck = splrep(t_mocap, knee_rad, s=0, k=3)
    return np.asarray(splev(t_mocap, tck, der=1), dtype=np.float64)


def build_vicon_replay_inputs(
    t_mocap: np.ndarray,
    knee_rad: np.ndarray,
    wave: Dict,
    fs_hz: float,
) -> Tuple[np.ndarray, np.ndarray]:
    knee_vel = _vicon_ik_velocity_spline(t_mocap, knee_rad)
    angle_sync = sync_to_wave_t(t_mocap, knee_rad, wave, t_src_on_npz_clock=False)
    vel_sync = sync_to_wave_t(t_mocap, knee_vel, wave, t_src_on_npz_clock=False)
    angle_lpf = apply_causal_lpf_series(angle_sync, fs_hz, VICON_ANGLE_LPF_HZ, VICON_ANGLE_LPF_ORDER)
    vel_lpf = apply_causal_lpf_series(vel_sync, fs_hz, TRAINING_VEL_LPF_HZ, TRAINING_VEL_LPF_ORDER)
    return angle_lpf, vel_lpf


def build_compare_wave(stem: str, wave: Dict, model: TCN, window_size: int, device: str) -> Dict:
    """Attach Vicon IK oracle; logged model output already in wave from batch §1."""
    out = dict(wave)
    t_aligned = np.asarray(wave['t'], dtype=np.float64)
    fs_hz = float(wave.get('fs_hz', infer_fs_hz(t_aligned)))

    t_mocap, knee_rad, _ = _load_vicon_knee_ik(stem)
    vicon_angle, vicon_vel = build_vicon_replay_inputs(t_mocap, knee_rad, wave, fs_hz)
    n = int(min(len(vicon_angle), len(out.get('gt_nmpkg', t_aligned)), len(out.get('model_out_nmpkg', t_aligned))))
    vicon_angle = vicon_angle[:n]
    vicon_vel = vicon_vel[:n]

    vicon_pred_raw = run_knee_tcn_inference(model, vicon_angle, vicon_vel, window_size, device).astype(np.float64)
    out['vicon_ik_model_out_nmpkg_raw'] = vicon_pred_raw
    out['vicon_ik_model_out_nmpkg'] = model_out_nmpkg_lpf(vicon_pred_raw, fs_hz)

    for key in ('t', 'gt_nmpkg', 'model_out_nmpkg', 'model_out_nmpkg_raw', 'vicon_ik_model_out_nmpkg_raw', 'vicon_ik_model_out_nmpkg'):
        if key in out and len(out[key]) > n:
            out[key] = np.asarray(out[key], dtype=np.float64)[:n]
    return out


print(f'Loading checkpoint: {VICON_ORACLE_CKPT}')
if not VICON_ORACLE_CKPT.is_file():
    raise FileNotFoundError(VICON_ORACLE_CKPT)

_ckpt = torch.load(str(VICON_ORACLE_CKPT), map_location='cpu', weights_only=False)
_model_cfg = _ckpt['model_config']
if _model_cfg.get('model_type', 'tcn') != 'tcn':
    raise ValueError(f"Expected TCN checkpoint, got {_model_cfg.get('model_type')!r}")

_window_size = int(_ckpt.get('window_size', _ctrl_cfg.get('frame_length', 100)))
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_oracle_model = TCN(**_tcn_ctor_kwargs(_model_cfg)).eval()
_oracle_model.load_state_dict(_ckpt['model_state_dict'])
_oracle_model.to(_device)
print(f'Vicon IK oracle ready | window={_window_size} | device={_device}')

COMPARE_DATA: Dict[str, Dict] = {}
_compare_errors = []
for stem, wave in sorted(TRIAL_DATA.items()):
    try:
        COMPARE_DATA[stem] = build_compare_wave(stem, wave, _oracle_model, _window_size, _device)
    except Exception as exc:
        _compare_errors.append((stem, str(exc)))

print(f'Built compare waveforms for {len(COMPARE_DATA)} / {len(TRIAL_DATA)} trials')
if _compare_errors:
    print('Skipped:')
    for stem, msg in _compare_errors:
        print(f'  {stem}: {msg}')

_compare_metrics_rows = []
for stem, wave in sorted(COMPARE_DATA.items()):
    meta = trial_meta_from_stem(stem)
    m = analysis_trim_mask(wave['t'])
    gt = wave['gt_nmpkg'][m]
    logged = wave['model_out_nmpkg'][m]
    vicon = wave['vicon_ik_model_out_nmpkg'][m]
    rmse_logged, r2_logged = rmse_r2(gt, logged)
    rmse_vicon, r2_vicon = rmse_r2(gt, vicon)
    _compare_metrics_rows.append({
        'trial': stem,
        'trial_key': meta['trial_key'],
        'subject': meta['subject'],
        'task': meta['task'],
        'condition': meta['condition'],
        'speed_mps': meta['speed_mps'],
        'rmse_logged_nmpkg': rmse_logged,
        'r2_logged_nmpkg': r2_logged,
        'rmse_vicon_oracle_nmpkg': rmse_vicon,
        'r2_vicon_oracle_nmpkg': r2_vicon,
        'rmse_logged_vs_vicon_nmpkg': float(np.sqrt(np.mean((logged - vicon) ** 2))),
        'model_out_key': wave.get('model_out_key', ''),
        'offset_s': wave['offset_s'],
    })

compare_metrics_df = pd.DataFrame(_compare_metrics_rows)
compare_metrics_df.to_csv(REPLAY_METRICS_CSV, index=False)
print(f'Saved metrics → {REPLAY_METRICS_CSV}')
display(compare_metrics_df)
for label, rmse_col, r2_col in [
    ('Logged output', 'rmse_logged_nmpkg', 'r2_logged_nmpkg'),
    ('Vicon IK oracle', 'rmse_vicon_oracle_nmpkg', 'r2_vicon_oracle_nmpkg'),
]:
    print(f"{label:16s}: RMSE={compare_metrics_df[rmse_col].mean():.4f} N·m/kg | R²={compare_metrics_df[r2_col].mean():.4f}")

compare_out = widgets.Output()
compare_trial_dd = widgets.Dropdown(options=sorted(COMPARE_DATA), description='Trial:')
compare_slider = widgets.FloatRangeSlider(
    description='Time (s):', continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='760px')
)


def draw_compare(wave: Dict, t_window: Tuple[float, float]) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    t0, t1 = t_window
    gt = wave['gt_nmpkg']
    logged = wave['model_out_nmpkg']
    vicon = wave['vicon_ik_model_out_nmpkg']
    m = (
        analysis_trim_mask(wave['t'])
        & (t_rel >= t0) & (t_rel <= t1)
        & np.isfinite(gt) & np.isfinite(logged) & np.isfinite(vicon)
    )
    rmse_logged, r2_logged = rmse_r2(gt[m], logged[m])
    rmse_vicon, r2_vicon = rmse_r2(gt[m], vicon[m])
    fig, axs = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axs[0].plot(t_rel[m], gt[m], color='#1e88e5', lw=2.0, label='GT (ID/mass − cmd/mass)')
    axs[0].plot(t_rel[m], logged[m], color='#fb8c00', lw=1.5, ls='-.', alpha=0.95, label='Logged output')
    axs[0].plot(t_rel[m], vicon[m], color='#43a047', lw=1.6, ls='-', label='Vicon IK oracle')
    axs[0].set_ylabel('N·m/kg')
    axs[0].set_title(
        f'Logged RMSE={rmse_logged:.4f} R²={r2_logged:.4f} | Vicon RMSE={rmse_vicon:.4f} R²={r2_vicon:.4f} | offset={wave["offset_s"]:+.3f}s'
    )
    axs[0].legend(loc='upper right', fontsize=8); axs[0].grid(alpha=0.25)
    axs[1].plot(t_rel[m], (logged - gt)[m], color='#fb8c00', lw=1.4, ls='-.', label='Logged − GT')
    axs[1].plot(t_rel[m], (vicon - gt)[m], color='#43a047', lw=1.4, label='Vicon − GT')
    axs[1].axhline(0, color='black', ls=':')
    axs[1].set_ylabel('Residual (N·m/kg)'); axs[1].set_xlabel('Time (s)')
    axs[1].legend(loc='upper right', fontsize=8); axs[1].grid(alpha=0.25)
    fig.tight_layout()
    with compare_out:
        compare_out.clear_output(wait=True)
        plt.show()


def _init_compare_slider(trial_key: str) -> None:
    wave = COMPARE_DATA[trial_key]
    t_rel = wave['t'] - np.nanmin(wave['t'])
    m = analysis_trim_mask(wave['t'])
    t_use = t_rel[m] if m.any() else t_rel
    compare_slider.min = float(t_use[0])
    compare_slider.max = float(t_use[-1])
    compare_slider.step = max((compare_slider.max - compare_slider.min) / 500, 1e-3)
    compare_slider.value = (compare_slider.min, compare_slider.max)


def _redraw_compare(*_):
    draw_compare(COMPARE_DATA[compare_trial_dd.value], compare_slider.value)


def _on_compare_trial(change):
    _init_compare_slider(change['new'])
    _redraw_compare()


compare_trial_dd.observe(_on_compare_trial, names='value')
compare_slider.observe(_redraw_compare, names='value')
if COMPARE_DATA:
    _init_compare_slider(compare_trial_dd.value)
    display(widgets.VBox([compare_trial_dd, compare_slider, compare_out]))
    _redraw_compare()
else:
    print('No trials to plot.')


## 4. Encoder vs Vicon IK knee angle

GPIO-synced comparison of on-board encoder vs `knee_angle_r` from processed IK.
For RD, §1 also shifts ID by the xcorr lag so GT matches encoder/logged timing (GPIO offset alone differs).


In [ ]:
if not TRIAL_DATA:
    raise RuntimeError('No trials loaded. Run batch processing first.')


def _vicon_ik_path(stem: str) -> Path:
    paths = resolve_trial_paths(stem)
    ik_path = paths['subject_dir'] / EXO_KIND / 'ik' / f"{paths['cond']}_{paths['speed']}_ik.mot"
    if not ik_path.is_file():
        raise FileNotFoundError(ik_path)
    return ik_path


def _load_vicon_knee_ik_deg(stem: str) -> Tuple[np.ndarray, np.ndarray]:
    cols, data = read_sto(_vicon_ik_path(stem))
    t_mocap = data[:, cols.index('time')].astype(np.float64)
    knee_deg = data[:, cols.index('knee_angle_r')].astype(np.float64)
    return t_mocap, knee_deg


def load_synced_encoder_angle_rad(stem: str, wave: Dict) -> Tuple[np.ndarray, str]:
    npz_path = TELEMETRY_ROOT / f'{stem}.npz'
    d = np.load(str(npz_path), allow_pickle=True)
    if 'time' not in d.files:
        raise KeyError(f"No 'time' in {npz_path.name}")
    t_npz = np.asarray(d['time'], dtype=np.float64)
    for key in ('model_in_knee_angle_raw', 'knee_angle_r'):
        if key in d.files:
            enc_raw = np.asarray(d[key], dtype=np.float64)
            enc_sync = sync_to_wave_t(t_npz, enc_raw, wave, t_src_on_npz_clock=True)
            return enc_sync, key
    raise KeyError(f'No encoder angle in {npz_path.name}; keys={sorted(d.files)}')


def load_synced_vicon_ik_angle_deg(stem: str, wave: Dict) -> np.ndarray:
    t_mocap, knee_deg = _load_vicon_knee_ik_deg(stem)
    return np.rad2deg(sync_to_wave_t(t_mocap, np.deg2rad(knee_deg), wave, t_src_on_npz_clock=False))


def _angle_xcorr_best_lag_samples(
    enc_deg: np.ndarray,
    vicon_deg: np.ndarray,
    *,
    max_lag: int = 300,
    mask: Optional[np.ndarray] = None,
) -> int:
    """Positive lag => encoder lags Vicon IK (shift encoder earlier)."""
    best_lag = 0
    best_score = -np.inf
    for lag in range(-int(max_lag), int(max_lag) + 1):
        shifted = _shift_samples_1d(enc_deg, lag)
        s = np.asarray(shifted, dtype=np.float64)
        v = np.asarray(vicon_deg, dtype=np.float64)
        if mask is not None:
            m = np.asarray(mask, dtype=bool)
            s, v = s[m], v[m]
        mm = np.isfinite(s) & np.isfinite(v)
        if mm.sum() < 100:
            continue
        score = float(np.corrcoef(s[mm], v[mm])[0, 1])
        if score > best_score:
            best_score = score
            best_lag = lag
    return int(best_lag)


def _pearson_r(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan
    return float(np.corrcoef(y_true[m], y_pred[m])[0, 1])


ANGLE_DATA: Dict[str, Dict] = {}
_angle_errors: List[Tuple[str, str]] = []

for stem, wave in sorted(TRIAL_DATA.items()):
    try:
        enc_rad, enc_key = load_synced_encoder_angle_rad(stem, wave)
        vicon_deg = load_synced_vicon_ik_angle_deg(stem, wave)
        n = int(min(len(enc_rad), len(vicon_deg), len(wave['t'])))
        t = np.asarray(wave['t'][:n], dtype=np.float64)
        enc_deg = np.rad2deg(enc_rad[:n])
        vicon_deg = vicon_deg[:n]
        trim_m = analysis_trim_mask(t)
        lag = _angle_xcorr_best_lag_samples(enc_deg, vicon_deg, mask=trim_m)
        enc_aligned_deg = _shift_samples_1d(enc_deg, lag)
        ANGLE_DATA[stem] = {
            'trial': stem,
            't': t,
            'encoder_deg': enc_deg,
            'encoder_aligned_deg': enc_aligned_deg,
            'vicon_ik_deg': vicon_deg,
            'encoder_key': enc_key,
            'xcorr_lag_samples': lag,
            'fs_hz': float(wave.get('fs_hz', infer_fs_hz(t))),
            'offset_s': float(wave['offset_s']),
        }
    except Exception as exc:
        _angle_errors.append((stem, str(exc)))

print(f'Built encoder/IK angle series for {len(ANGLE_DATA)} / {len(TRIAL_DATA)} trials')
if _angle_errors:
    print('Skipped:')
    for stem, msg in _angle_errors:
        print(f'  {stem}: {msg}')

_angle_metrics_rows = []
for stem, d in sorted(ANGLE_DATA.items()):
    meta = trial_meta_from_stem(stem)
    m = analysis_trim_mask(d['t'])
    enc = d['encoder_deg'][m]
    enc_al = d['encoder_aligned_deg'][m]
    vicon = d['vicon_ik_deg'][m]
    rmse_raw, r2_raw = rmse_r2(vicon, enc)
    rmse_al, r2_al = rmse_r2(vicon, enc_al)
    fs_hz = d['fs_hz']
    lag = int(d['xcorr_lag_samples'])
    _angle_metrics_rows.append({
        'trial': stem,
        'trial_key': meta['trial_key'],
        'task': meta['task'],
        'condition': meta['condition'],
        'encoder_key': d['encoder_key'],
        'rmse_encoder_deg': rmse_raw,
        'r2_encoder_deg': r2_raw,
        'pearson_encoder': _pearson_r(vicon, enc),
        'rmse_encoder_aligned_deg': rmse_al,
        'r2_encoder_aligned_deg': r2_al,
        'pearson_encoder_aligned': _pearson_r(vicon, enc_al),
        'xcorr_lag_samples': lag,
        'xcorr_lag_ms': lag / fs_hz * 1000.0,
        'offset_s': d['offset_s'],
    })

angle_metrics_df = pd.DataFrame(_angle_metrics_rows)
display(angle_metrics_df)
if len(angle_metrics_df):
    print(
        f"Encoder vs IK (trimmed): RMSE={angle_metrics_df['rmse_encoder_deg'].mean():.2f}° | "
        f"R²={angle_metrics_df['r2_encoder_deg'].mean():.3f} | "
        f"r={angle_metrics_df['pearson_encoder'].mean():.3f}"
    )
    print(
        f"After xcorr lag align: RMSE={angle_metrics_df['rmse_encoder_aligned_deg'].mean():.2f}° | "
        f"R²={angle_metrics_df['r2_encoder_aligned_deg'].mean():.3f} | "
        f"r={angle_metrics_df['pearson_encoder_aligned'].mean():.3f}"
    )

angle_out = widgets.Output()
angle_trial_dd = widgets.Dropdown(options=sorted(ANGLE_DATA), description='Trial:')
angle_slider = widgets.FloatRangeSlider(
    description='Time (s):', continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='760px')
)
angle_show_aligned = widgets.Checkbox(value=True, description='Show xcorr-aligned encoder')


def draw_encoder_ik_angle(d: Dict, t_window: Tuple[float, float], show_aligned: bool) -> None:
    t_rel = d['t'] - np.nanmin(d['t'])
    t0, t1 = t_window
    m = analysis_trim_mask(d['t']) & (t_rel >= t0) & (t_rel <= t1)
    vicon = d['vicon_ik_deg'][m]
    enc = d['encoder_deg'][m]
    enc_al = d['encoder_aligned_deg'][m]
    rmse_raw, r2_raw = rmse_r2(vicon, enc)
    rmse_al, r2_al = rmse_r2(vicon, enc_al)
    lag = int(d['xcorr_lag_samples'])
    lag_ms = lag / d['fs_hz'] * 1000.0

    fig, axs = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axs[0].plot(t_rel[m], vicon, color='#43a047', lw=1.6, label='Vicon IK knee_angle_r')
    axs[0].plot(t_rel[m], enc, color='#fb8c00', lw=1.4, ls='--', alpha=0.9, label=f"Encoder ({d['encoder_key']})")
    if show_aligned:
        axs[0].plot(t_rel[m], enc_al, color='#e65100', lw=1.2, ls='-.', alpha=0.95, label='Encoder (xcorr aligned)')
    axs[0].set_ylabel('Knee angle (deg)')
    axs[0].set_title(
        f"Raw RMSE={rmse_raw:.2f}° R²={r2_raw:.3f} | aligned RMSE={rmse_al:.2f}° R²={r2_al:.3f} | lag={lag:+d} samples ({lag_ms:+.0f} ms)"
    )
    axs[0].legend(loc='upper right', fontsize=8)
    axs[0].grid(alpha=0.25)

    resid_enc = enc - vicon
    axs[1].plot(t_rel[m], resid_enc, color='#fb8c00', lw=1.3, ls='--', label='Encoder − Vicon')
    if show_aligned:
        axs[1].plot(t_rel[m], enc_al - vicon, color='#e65100', lw=1.2, ls='-.', label='Aligned encoder − Vicon')
    axs[1].axhline(0, color='black', ls=':')
    axs[1].set_ylabel('Residual (deg)')
    axs[1].set_xlabel('Time (s)')
    axs[1].legend(loc='upper right', fontsize=8)
    axs[1].grid(alpha=0.25)
    fig.tight_layout()
    with angle_out:
        angle_out.clear_output(wait=True)
        plt.show()


def _init_angle_slider(trial_key: str) -> None:
    d = ANGLE_DATA[trial_key]
    t_rel = d['t'] - np.nanmin(d['t'])
    m = analysis_trim_mask(d['t'])
    t_use = t_rel[m] if m.any() else t_rel
    angle_slider.min = float(t_use[0])
    angle_slider.max = float(t_use[-1])
    angle_slider.step = max((angle_slider.max - angle_slider.min) / 500, 1e-3)
    angle_slider.value = (angle_slider.min, angle_slider.max)


def _redraw_angle(*_):
    draw_encoder_ik_angle(ANGLE_DATA[angle_trial_dd.value], angle_slider.value, angle_show_aligned.value)


def _on_angle_trial(change):
    _init_angle_slider(change['new'])
    _redraw_angle()


angle_trial_dd.observe(_on_angle_trial, names='value')
angle_slider.observe(_redraw_angle, names='value')
angle_show_aligned.observe(_redraw_angle, names='value')
if ANGLE_DATA:
    _init_angle_slider(angle_trial_dd.value)
    display(widgets.VBox([widgets.HBox([angle_trial_dd, angle_show_aligned]), angle_slider, angle_out]))
    _redraw_angle()
else:
    print('No encoder/IK angle trials to plot.')
